In [1]:
import pyhidra

# FYI: this "initializes the application" (per Ghidra docs)
launcher = pyhidra.start()

In [2]:
#!echo $PATH | tr ':' '\n'
import typing
if typing.TYPE_CHECKING:
    import ghidra
    from ghidra.ghidra_builtins import *

import ghidra
from ghidra.base.project import GhidraProject

# ghidra://localhost/binutils/run1.gcc-O0.binutils-2_36
# repo.fileExists('/run1.gcc-O0.binutils-2_36', '0.gdb.debug')

host = 'localhost'
port = 13100
repoName = 'astera'
folderPath = '/run1.gcc.astera'
binaryName = '0.fighter.debug'


In [3]:
from ghidralib import OpenSharedGhidraProject

from ghidra.program.model.pcode import PcodeBlockBasic
from ghidra.app.decompiler import *
# from ghidra.app.decompiler import DecompInterface, DecompileOptions
from ghidralib import get_decompiler_interface, AstBuilder

DECOMPILE_TIMEOUT_SEC = 180

with OpenSharedGhidraProject(host, repoName, port) as proj:

    prog = proj.openProgram(folderPath, binaryName, True)
    fm = prog.getFunctionManager()
    nonthunks = (x for x in fm.getFunctions(True) if not x.isThunk())

    test_func = [x for x in nonthunks if x.name == 'main'][0]
    print(test_func.name)

    ifc = get_decompiler_interface(prog)
    res = ifc.decompileFunction(test_func, DECOMPILE_TIMEOUT_SEC, None)
    tudecl = AstBuilder(res).build_func_ast()
tudecl

main
Python 3.8.10 (default, May 26 2023, 14:05:08) 
Type 'copyright', 'credits' or 'license' for more information
IPython 8.12.3 -- An enhanced Interactive Python. Type '?' for help.

Out[1]: <ghidralib.astbuilder.AstBuilder at 0x7fd1bf564af0>

Out[2]: ghidra.app.decompiler.DecompileResults@7c2e88b9




In [5]:

def remove_comments_and_blank_lines(ghidra_c:str) -> str:
    '''
    Remove comment lines and blank lines from Ghidra C code to help our AST diff match
    '''
    return '\n'.join([l for l in ghidra_c.split('\n') if l.strip() and l.strip()[:2] != '/*'])

with open(f'{test_func.name}.ghidra.c', 'w') as f:
    f.write(remove_comments_and_blank_lines(res.getDecompiledFunction().getC()))

In [6]:
# TODO: write my version..

with open(f'{test_func.name}.ast.c', 'w') as f:
    f.write('\n/* WARNING: Removing unreachable block (ram,0x0014e339) */')